# Landing outcomes and data wrangling

Applied Data Science Capstone. Rodrigo Ariel Quilodran.

Historical IBM course datasets, analysis executed 7 November 2026. AI-assisted preparation; source data and limitations are disclosed.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA, OUT, FIG = ROOT/'data', ROOT/'results', ROOT/'figures'
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False})


## Data inspection and counts

In [2]:
df=pd.read_csv(OUT/'collected_launches.csv')
print(df.shape)
display(df.isna().sum().to_frame('Missing'))
display(df.LaunchSite.value_counts().to_frame('Launches'))
display(df.Orbit.value_counts().to_frame('Launches'))
display(df.Outcome.value_counts().to_frame('Launches'))

(90, 17)


,Missing
FlightNumber,0
Date,0
BoosterVersion,0
PayloadMass,0
Orbit,0
LaunchSite,0
Outcome,0
Flights,0
GridFins,0
Reused,0


,Launches
LaunchSite,
CCAFS SLC 40,55
KSC LC 39A,22
VAFB SLC 4E,13


,Launches
Orbit,
GTO,27
ISS,21
VLEO,14
PO,9
LEO,7
SSO,5
MEO,3
ES-L1,1
HEO,1


,Launches
Outcome,
True ASDS,41
None None,19
True RTLS,14
False ASDS,6
True Ocean,5
False Ocean,2
None ASDS,2
False RTLS,1


## Binary target
Class=1 denotes an observed successful landing (True ASDS, True RTLS or True Ocean). Class=0 combines failed, unknown and unattempted outcomes as required by the course. A successful ocean landing is not necessarily a recovered reusable booster.

In [3]:
bad={'False ASDS','False Ocean','False RTLS','None ASDS','None None'}
df['Class']=(~df.Outcome.isin(bad)).astype(int)
ref=pd.read_csv(DATA/'dataset_part_2.csv')
assert df.Class.equals(ref.Class)
print('Successful:',df.Class.sum(),'Total:',len(df),'Rate:',df.Class.mean())
print('GTO launches:',df.Orbit.eq('GTO').sum())
print('Successful drone-ship landings:',df.Outcome.eq('True ASDS').sum())
df.to_csv(OUT/'dataset_part_2.csv',index=False)
summary={'rows':len(df),'successes':int(df.Class.sum()),'success_rate':float(df.Class.mean()),'sites':df.groupby('LaunchSite').Class.agg(['count','sum','mean']).reset_index().to_dict('records'),'orbits':df.groupby('Orbit').Class.agg(['count','sum','mean']).reset_index().to_dict('records'),'years':df.assign(Year=pd.to_datetime(df.Date).dt.year).groupby('Year').Class.agg(['count','sum','mean']).reset_index().to_dict('records')}
(OUT/'summary.json').write_text(json.dumps(summary,indent=2))
display(df.head())

Successful: 60 Total: 90 Rate: 0.6666666666666666
GTO launches: 27
Successful drone-ship landings: 41


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,0
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,0
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,0
